## Row filters

A **row filter** is a function that returns a boolean per row; Unity Catalog adds an implicit `WHERE` on every read, keeping only rows where the function returns `TRUE` for the querying principal.

```sql
CREATE OR REPLACE FUNCTION fintech_dev.security.region_filter(country STRING)
RETURNS BOOLEAN
RETURN CASE
  WHEN is_account_group_member('compliance_in') THEN country = 'IN'
  WHEN is_account_group_member('compliance_us') THEN country = 'US'
  WHEN is_account_group_member('compliance')    THEN TRUE
  ELSE FALSE
END;

ALTER TABLE fintech_dev.silver.customers
  SET ROW FILTER fintech_dev.security.region_filter ON (country);
```

Now every `SELECT FROM silver.customers` is implicitly `WHERE region_filter(country) = TRUE` — each principal sees only their slice. A member of neither group matches the `ELSE FALSE` branch and sees **nothing**.

**Remove it:** `ALTER TABLE ... DROP ROW FILTER;`.

**Column mask + row filter = the bank's full PII regime.** A compliance analyst in India sees full PII, but only `country = 'IN'` rows. A plain analyst sees only `IN` rows *and* gets `REDACTED` in the PAN column. Masks control **which columns' values** you see; row filters control **which rows** you see — combine them for column-and-row-level security on one base table.
